# Inicialization 

In [ ]:
import copy
import json
import os

import matplotlib.pyplot as plt
import numpy as np
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

from students.ushatov.lesson3 import Exercise as ex

## Dataset

In [ ]:
X, y = load_digits(return_X_y=True)

X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

X_train, X_val, y_train, y_val = train_test_split(
    X_train_val, y_train_val, test_size=0.25, random_state=42, stratify=y_train_val
)

In [ ]:
# Нормализация
scaler = StandardScaler()
X_train_norm = scaler.fit_transform(X_train)
X_val_norm = scaler.transform(X_val)
X_test_norm = scaler.transform(X_test)

## Metric


In [ ]:
loss_train = []

## Train

In [ ]:
def train(model, loss, x: np.ndarray, y: np.ndarray, lr: float, batch_size: int):
    n = x.shape[0]
    batch = n if (batch_size is None or batch_size >= n) else batch_size

    for count, start in enumerate(range(0, n, batch)):
        end = min(start + batch, n)
        x_batch = x[start:end]
        y_batch = y[start:end]

        predict = model.forward(x_batch)
        loss_batch = loss.forward(predict, y_batch)

        model.backward(loss.backward())

        # Обновление параметров
        for param, grad in zip(model.parameters, model.grad, strict=True):
            param -= lr * grad

        if count % 10 == 0:
            loss_train.append(loss_batch)

## Test

In [ ]:
def test(model, X, y):
    log_probs = model.forward(X)
    predicts = np.argmax(log_probs, axis=1)
    return np.mean(predicts == y)

# Learning

## Constant


In [ ]:
IN_LOYER = 8 * 8
OUT_LOYER = 10

## Learn model

In [ ]:
def learn(first, epochs, lr, batch):
    rng = np.random.default_rng(42)

    model = ex.create_model(
        ex.create_linear_layer(IN_LOYER, first, rng),
        ex.create_relu_layer(),
        ex.create_linear_layer(first, OUT_LOYER, rng),
    )

    loss_fn = ex.create_cross_entropy_loss()

    accuracies = []
    for epoch in range(1, epochs + 1):
        train(model, loss_fn, X_train_norm, y_train, lr, batch)
        acc = test(model, X_val_norm, y_val)
        accuracies.append(acc)
        acct = test(model, X_test_norm, y_test)
        print(f"Эпоха {epoch}: val_acc = {acc:.4f} test_acc = {acct:.4f}   {acc - acct:4f}")

    print(f"\nФинальная точность на валидации: {accuracies[-1]:.4f}")
    print(f"Точность на тесте: {test(model, X_test_norm, y_test):.4f}")

In [ ]:
learn(174, 41, 0.451762, 56)

In [ ]:
learn(100, 16, 0.7081, 58)

In [ ]:
learn(194, 41, 0.5440, 38)

In [ ]:
learn(181, 37, 0.8696, 125)

# SL Finding the optimal epoch for the initial search of hyperparameters

In [ ]:
# Параметры генерации
num_comb = 200

# Диапазоны
lr1 = (8, 256)  # first_layer_range
lr2 = (8, 256)  # second_layer_range
lr3 = (8, 256)  # third_layer_range
lrr = (0.001, 1)
batchr = (16, 128)
epoch = 100

rng = np.random.default_rng(42)

In [ ]:
def optuna_3layer(
    first_layer_range, second_layer_range, third_layer_range, epochs, lr_range, batch_range, rng, num_combinations
):

    res = {}
    for i in range(1, num_combinations, 1):
        fsize = rng.integers(first_layer_range[0], first_layer_range[1] + 1, 1)[0]
        ssize = rng.integers(second_layer_range[0], second_layer_range[1] + 1, 1)[0]
        tsize = rng.integers(third_layer_range[0], third_layer_range[1] + 1, 1)[0]

        log_lr = rng.uniform(np.log(lr_range[0]), np.log(lr_range[1]), 1)[0]
        lr = np.exp(log_lr)
        bs = rng.integers(batch_range[0], batch_range[1] + 1, 1)[0]

        model = ex.create_model(
            ex.create_linear_layer(IN_LOYER, fsize, rng),
            ex.create_relu_layer(),
            ex.create_linear_layer(fsize, ssize, rng),
            ex.create_relu_layer(),
            ex.create_linear_layer(ssize, tsize, rng),
            ex.create_relu_layer(),
            ex.create_linear_layer(tsize, OUT_LOYER, rng),
        )

        loss_fn = ex.create_cross_entropy_loss()

        # Список для сохранения accuracy на каждой эпохе
        val_accuracies = []
        for _ in range(1, epochs + 1):
            train(model, loss_fn, X_train_norm, y_train, lr, bs)
            accuracy = test(model, X_val_norm, y_val)
            val_accuracies.append(accuracy)

        key = (fsize, ssize, tsize, lr, bs)
        res[key] = val_accuracies
        print(
            f"{i}.fsize={fsize}, ssize={ssize}, tsize={tsize}, lr={lr:.4f}, bs={bs} | accuracy={val_accuracies[-1]:.4f}"
        )
    return res

In [ ]:
def optuna_2layer(first_layer_range, second_layer_range, epochs, lr_range, batch_range, rng, num_combinations):

    res = {}
    for i in range(1, num_combinations, 1):
        fsize = rng.integers(first_layer_range[0], first_layer_range[1] + 1, 1)[0]
        ssize = rng.integers(second_layer_range[0], second_layer_range[1] + 1, 1)[0]
        log_lr = rng.uniform(np.log(lr_range[0]), np.log(lr_range[1]), 1)[0]
        lr = np.exp(log_lr)
        bs = rng.integers(batch_range[0], batch_range[1] + 1, 1)[0]

        model = ex.create_model(
            ex.create_linear_layer(IN_LOYER, fsize, rng),
            ex.create_relu_layer(),
            ex.create_linear_layer(fsize, ssize, rng),
            ex.create_relu_layer(),
            ex.create_linear_layer(ssize, OUT_LOYER, rng),
        )

        loss_fn = ex.create_cross_entropy_loss()

        # Список для сохранения accuracy на каждой эпохе
        val_accuracies = []
        for _ in range(1, epochs + 1):
            train(model, loss_fn, X_train_norm, y_train, lr, bs)
            accuracy = test(model, X_val_norm, y_val)
            val_accuracies.append(accuracy)

        key = (fsize, ssize, lr, bs)
        res[key] = val_accuracies
        print(f"{i}. fsize={fsize}, ssize={ssize}, lr={lr:.4f}, bs={bs} | accuracy={val_accuracies[-1]:.4f}")
    return res

In [ ]:
def optuna_1layer(first_layer_range, epochs, lr_range, batch_range, rng, num_combinations):

    res = {}
    for i in range(1, num_combinations, 1):
        rng = np.random.default_rng(i)
        fsize = rng.integers(first_layer_range[0], first_layer_range[1] + 1, 1)[0]
        log_lr = rng.uniform(np.log(lr_range[0]), np.log(lr_range[1]), 1)[0]
        lr = np.exp(log_lr)
        bs = rng.integers(batch_range[0], batch_range[1] + 1, 1)[0]

        model = ex.create_model(
            ex.create_linear_layer(IN_LOYER, fsize, rng),
            ex.create_relu_layer(),
            ex.create_linear_layer(fsize, OUT_LOYER, rng),
        )
        loss_fn = ex.create_cross_entropy_loss()

        val_accuracies = []
        for _ in range(1, epochs + 1):
            train(model, loss_fn, X_train_norm, y_train, lr, bs)
            accuracy = test(model, X_val_norm, y_val)
            val_accuracies.append(accuracy)

        key = (fsize, lr, bs)
        res[key] = val_accuracies
        print(f"{i}. fsize={fsize}, lr={lr:.4f}, bs={bs} | accuracy={val_accuracies[-1]:.4f}")
    return res

In [ ]:
results_1 = optuna_1layer(lr1, epoch, lrr, batchr, rng, num_comb)

In [ ]:
results_2 = optuna_2layer(lr1, lr2, epoch, lrr, batchr, rng, num_comb)

In [ ]:
results_3 = optuna_3layer(lr1, lr2, lr3, epoch, lrr, batchr, rng, num_comb)

## Save data

In [ ]:
def save_data(results, filename: str, save_dir="students/ushatov/digits_data"):

    filename_txt = os.path.join(save_dir, filename)

    results_str_keys = {str(k): v for k, v in results.items()}

    with open(filename_txt, "w") as file:
        json.dump(results_str_keys, file, indent=4, ensure_ascii=False)

    print(f"Результаты сохранены в {filename_txt}")

In [ ]:
save_data(results_1, "digits_training_results_1.txt")

In [ ]:
save_data(results_2, "digits_training_results_2.txt")

In [ ]:
save_data(results_3, "digits_training_results_3.txt")

# Result


In [ ]:
def load(where: str):

    with open(where) as file:
        data_str = file.read()

    data_str_clean = data_str.replace("np.int64(", "").replace("np.float64(", "")
    data = eval(data_str_clean)

    series_list = list(data.values())
    min_len = min(len(s) for s in series_list)
    truncated = np.array([s[:min_len] for s in series_list])
    n_exp, n_epochs = truncated.shape
    print(f"Всего экспериментов: {n_exp}, эпох: {n_epochs}")

    # Топ-20 по эпохам
    top_k = 20 if n_exp >= 20 else n_exp
    top_sets = []
    for epoch in range(n_epochs):
        metrics = truncated[:, epoch]
        sorted_idx = np.argsort(metrics)[::-1]
        top_sets.append(set(sorted_idx[:top_k]))

    last_epoch = n_epochs - 1
    top_last = top_sets[last_epoch]

    # Количество общих моделей с финальным топ-20 на каждой эпохе
    intersection_counts = [len(top_sets[epoch].intersection(top_last)) for epoch in range(n_epochs)]

    # Накопленные уникальные модели, которые хоть раз попали в топ-20 и остались в финале
    cumulative_unique = []
    cumulative_set = set()
    for epoch in range(n_epochs):
        cumulative_set.update(top_sets[epoch].intersection(top_last))
        cumulative_unique.append(len(cumulative_set))

    # ГРАФИК 1: ДИНАМИКА ТОП-20
    plt.figure(figsize=(12, 6))
    plt.plot(
        range(n_epochs), intersection_counts, marker="o", linestyle="-", color="green", label="Общие с финальным топ-20"
    )
    plt.plot(
        range(n_epochs), cumulative_unique, marker="s", linestyle="--", color="blue", label="Накопленные уникальные"
    )
    plt.xlabel("Номер эпохи")
    plt.ylabel("Количество моделей")
    plt.title("Динамика топ-20: общие с финальным топ-20, накопленные уникальные")
    plt.ylim(0, top_k + 1)
    plt.grid(True, alpha=0.3)
    plt.legend()
    plt.tight_layout()
    plt.show()

    # ГРАФИК 2: СРЕДНЕЕ ИЗМЕНЕНИЕ МЕТРИКИ
    diffs = np.abs(np.diff(truncated, axis=1))
    avg_abs_change = np.mean(diffs, axis=0)

    plt.figure(figsize=(10, 5))
    plt.plot(range(1, n_epochs), avg_abs_change, marker="o", linestyle="-", linewidth=1, markersize=3)
    plt.xlabel("Шаг (эпоха t → t+1)")
    plt.ylabel("Среднее абсолютное изменение метрики")
    plt.title("Средний модуль изменения метрики от эпохи к эпохе")
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

    # ТЕПЛОВАЯ КАРТА (финальный ранг на позициях текущего топ-20)
    metrics_last = truncated[:, last_epoch]
    sorted_indices_last = np.argsort(metrics_last)[::-1]
    final_top_indices = sorted_indices_last[:20]
    final_rank = {idx: i + 1 for i, idx in enumerate(final_top_indices)}  # 1..20

    top20_per_epoch = []
    for epoch in range(n_epochs):
        metrics = truncated[:, epoch]
        sorted_idx = np.argsort(metrics)[::-1]
        top20_per_epoch.append(sorted_idx[:20])

    data_matrix = np.zeros((20, n_epochs), dtype=int)
    for epoch in range(n_epochs):
        for pos, model_idx in enumerate(top20_per_epoch[epoch]):
            data_matrix[pos, epoch] = final_rank.get(model_idx, 0)

    fig, ax = plt.subplots(figsize=(n_epochs * 0.3, 6))
    cmap = plt.get_cmap("RdYlGn_r")
    cmap.set_bad("white")
    masked = np.ma.masked_where(data_matrix == 0, data_matrix)

    # Вместо Normalize используем vmin/vmax напрямую в imshow
    im = ax.imshow(masked, cmap=cmap, vmin=1, vmax=20, aspect="auto", interpolation="nearest")

    ax.set_xticks(np.arange(n_epochs))
    ax.set_yticks(np.arange(20))
    ax.set_xticklabels([str(i) for i in range(1, n_epochs + 1)], rotation=90, fontsize=6)
    ax.set_yticklabels([str(i) for i in range(1, 21)], fontsize=7)
    ax.set_xlabel("Эпоха", fontsize=10)
    ax.set_ylabel("Место в топ-20 текущей эпохи (1 – лучший)", fontsize=10)
    ax.set_title(
        "Финальный ранг модели на каждой позиции текущего топ-20\n"
        "(цвет: 1=лучший в конце, 20=20-й в конце, белый=вне финального топ-20)",
        fontsize=12,
    )

    cbar = fig.colorbar(im, ax=ax, ticks=range(1, 21))
    cbar.set_label("Финальный ранг модели", rotation=270, labelpad=15)

    plt.tight_layout()
    plt.show()

In [ ]:
load("students/ushatov/digits_data/digits_training_results_1.txt")

In [ ]:
load("students/ushatov/digits_data/digits_training_results_2.txt")

In [ ]:
load("students/ushatov/digits_data/digits_training_results_3.txt")

# SL2

In [ ]:
# ПАРАМЕТРЫ
IN_LAYER = 64
OUT_LAYER = 10
SAVE_EPOCH = 15
MAX_EPOCHS_TOTAL = 90

num_combinations = 1000
first_layer_range = (32, 200)
lr_range = (0.001, 1)
batch_range = (16, 128)

LR_DECAY_FACTOR = 1.0

rng = np.random.default_rng(42)

In [ ]:
# ОБУЧЕНИЕ ВСЕХ МОДЕЛЕЙ ДО SAVE_EPOCH
results_all = {}
models_state_all = {}


for i in range(1, num_combinations + 1, 1):
    fsize = rng.integers(first_layer_range[0], first_layer_range[1] + 1)
    log_lr = rng.uniform(np.log(lr_range[0]), np.log(lr_range[1]))
    lr = np.exp(log_lr)
    bs = rng.integers(batch_range[0], batch_range[1] + 1)

    rng_inizialization = np.random.default_rng(42)
    model = ex.create_model(
        ex.create_linear_layer(IN_LAYER, int(fsize), rng_inizialization),
        ex.create_relu_layer(),
        ex.create_linear_layer(int(fsize), OUT_LAYER, rng_inizialization),
    )

    loss_fn = ex.create_cross_entropy_loss()

    val_accuracies = []
    for _epoch in range(1, SAVE_EPOCH + 1):
        train(model, loss_fn, X_train_norm, y_train, lr, int(bs))
        acc = test(model, X_val_norm, y_val)
        val_accuracies.append(acc)

    model_copy = copy.deepcopy(model)
    key = (int(fsize), float(lr), int(bs))
    results_all[key] = val_accuracies
    models_state_all[key] = {"model": model_copy, "lr": lr, "bs": bs, "rng_state": rng.bit_generator.state}
    print(f"{i}. fsize={key[0]}, lr={key[1]:.4f}, bs={key[2]} | accp={val_accuracies[-1]:.4f}")

# ВЗЯТИК МОДЕЛЕЙ БЫВШИХ В ТОП 20 ДО SAVE_EPOCH
top_models_set = set()
for epoch_idx in range(SAVE_EPOCH):  # эпоха 1 idx=0
    epoch_accs = {key: acc_list[epoch_idx] for key, acc_list in results_all.items()}

    sorted_keys = sorted(epoch_accs.items(), key=lambda x: x[1], reverse=True)
    top_20_keys = [k for k, _ in sorted_keys[:20]]
    top_models_set.update(top_20_keys)

print(f"\nВсего моделей, побывавших в топ-20 хотя бы раз на эпохах 1..{SAVE_EPOCH}: {len(top_models_set)}")

# ДООБУЧЕНИЕ МОДЕЛЕЙ
full_results = {}  # полная история точности
best_global_acc = -np.inf
best_global_model = None
best_global_epoch = None

for key in top_models_set:
    state = models_state_all[key]
    model = state["model"]
    lr = state["lr"] * LR_DECAY_FACTOR
    bs = state["bs"]
    loss_fn = ex.create_cross_entropy_loss()

    history = results_all[key].copy()
    # Дообучаем
    for epoch in range(SAVE_EPOCH + 1, MAX_EPOCHS_TOTAL + 1):
        train(model, loss_fn, X_train_norm, y_train, lr, bs)
        acc = test(model, X_val_norm, y_val)
        history.append(acc)
        if acc > best_global_acc:
            best_global_acc = acc
            best_global_model = key
            best_global_epoch = epoch
    full_results[key] = history
    print(f"Дообучено fsize={key[0]}, lr={key[1]:.4f}, bs={key[2]}: acc={history[-1]:.4f}, best={max(history):.4f}")

# ВЫВОД ИТОГОВ
print("\n=== ГЛОБАЛЬНО ЛУЧШАЯ МОДЕЛЬ ===")
if best_global_model is not None:
    fsize, lr, bs = best_global_model
    print(f"Гиперпараметры: fsize={fsize}, lr={lr:.6f}, bs={bs}")
    print(f"Максимальная accuracy: {best_global_acc:.4f}")
    print(f"Достигнута на эпохе: {best_global_epoch}")